In [ ]:
import sys
!{sys.executable} -m pip install webdriver-manager

In [11]:
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import os
import re
from webdriver_manager.chrome import ChromeDriverManager
import time
import pprint
from lxml import html
import random

In [12]:
def requestPage(link):
    try:
        response = requests.get(link)
        response.raise_for_status()  # Raise an exception for 4xx or 5xx status codes
        return response.text  # Return the HTML content of the page
    except requests.RequestException as e:
        print(f"Error fetching page: {e}")
        return None


In [13]:
def requestPageUsingWebDriver(self, link, web_agent="chrome"):
   options = driver = None
   if web_agent == "firefox":
       options = webdriver.FirefoxOptions()
       options.add_argument("--headless")
       driver = webdriver.Firefox(options=options)
   elif web_agent == "chrome":
       options = Options()
       options.add_argument("--disable-dev-shm-usage")
       options.add_argument("--headless")
       options.add_argument("--no-sandbox")
       driver = webdriver.Chrome(options=options)
   else:
       raise Exception("Invalid Web Agent! Please use firefox or chrome.")
   driver.get(link)
   driver.implicitly_wait(50)
   soup = BeautifulSoup(driver.page_source, "html.parser")
   driver.quit()
   return soup



In [14]:
def requestPage(link):
    try:
        print(f"Fetching page: {link}")
        response = requests.get(link)
        response.raise_for_status()  # Raise an exception for 4xx or 5xx status codes
        soup = BeautifulSoup(response.text, 'html.parser')  # Parse the HTML content
        return soup
    except requests.RequestException as e:
        print(f"Error fetching page: {e}")
        return None


In [15]:
def extractArticleLinks(category_link, show_more,max_links=7000,sections=['news']):
    article_links = []
    domain_url = 'https://www.aljazeera.com/tag/israel-palestine-conflict/'
    news_regex = re.compile(r'\/news\/\d+')
    
    options = Options()
    driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()))
    wait = WebDriverWait(driver, 2)  # Adjust the timeout as needed
    
    driver.get(category_link)
    for i in range(show_more):
        try:
            time.sleep(3)
            show_more_button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "show-more-button.big-margin")))
                        
            #show_more_button = driver.find_element(By.CLASS_NAME, "screen-reader-text")
            driver.execute_script("arguments[0].click();", show_more_button)
        except Exception as e:
            print("No more 'Show More' buttons to click.")
            break
    
    #while len(article_links) < max_links:
    article_tags = driver.find_elements(By.CSS_SELECTOR, "a.u-clickable-card__link")
    for a_tag in article_tags:
        link = a_tag.get_attribute("href")
        if any(section in link for section in sections):
            if not link.startswith("http"):
                link = domain_url + link
            if news_regex.search(link):
                article_links.append(link)
                if len(article_links) >= max_links:
                    break


    driver.quit()  # Close the WebDriver session
    
    return article_links[:max_links]  # Return only up to max_links


In [19]:
category_link = "https://www.aljazeera.com/tag/israel-palestine-conflict/"
show_more = 800  # quante volte cliccare "Show more"

links = extractArticleLinks(
    category_link=category_link,
    show_more=show_more,
    max_links=3000,
    sections=['news']
)

len(links)

No more 'Show More' buttons to click.


2277

In [20]:
links[-1]

'https://www.aljazeera.com/news/2024/9/9/why-has-israel-closed-the-west-banks-crossing-with-jordan'

In [26]:
df = pd.DataFrame({"url": sorted(links)})

In [28]:
df.to_csv(
    "articles_links_9-9-24.csv",
    index=False
)

In [ ]:
data = pd.read_csv("articles_links_9-9-24.csv")

In [ ]:
def requestPage(url):
    try:
        response = requests.get(
            url,
            headers={'User-Agent': random.choice(user_agents_list)},
            timeout=10
        )
        if response.status_code == 200:
            return BeautifulSoup(response.content, 'html.parser')
    except Exception:
        return None

In [ ]:
user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36'
]

def get_article(url):
    page = requests.get(url, headers={'User-Agent': random.choice(user_agents_list)})
    soup  = BeautifulSoup(page.content, 'html.parser')
    article = soup.find_all('p')
    return article

In [ ]:
index = 0
l= []
for url in data['url'].values:
    index += 1
    if index % 10 == 0:
        print(index)
    l.append(get_article(url))

In [ ]:
# Function to scrape an individual article
def scrapeArticle(link):
    parsed_page = requestPage(link)
    if parsed_page:
        title_element = parsed_page.find('h1', class_='article-heading__title')
        if title_element:
            title = title_element.text.strip()
        else:
            title = "No Title"

        author_element = parsed_page.find('a', class_='author-link')
        if author_element:
            author = author_element.text.strip()
        else:
            author = "Unknown"

        date_element = parsed_page.find('time', class_='date__time')
        if date_element and 'datetime' in date_element.attrs:
            date = date_element['datetime']
        else:
            date = "Unknown"

        content_element = parsed_page.find('div', class_='wysiwyg')
        if content_element:
            paragraphs = [p.text.strip() for p in content_element.find_all('p')]
            content = '\n'.join(paragraphs)
        else:
            content = "No Content"
        
        # Create a dictionary to store the scraped data
        article_data = {
            'Title': title,
            'Author': author,
            'Date': date,
            'Link': link,
            'Content': content
        }
        
        return article_data
    else:
        print(f"Failed to scrape article: {link}")
        return None


In [ ]:
def extract_date_from_url(url):
    # Define the regular expression pattern to match the date format in the URL
    date_pattern = r'(\d{4})/(\d{1,2})/(\d{1,2})/'
    
    # Search for the date pattern in the URL
    match = re.search(date_pattern, url)
    
    if match:
        # Extract year, month, and day from the matched groups
        year = int(match.group(1))
        month = int(match.group(2))
        day = int(match.group(3))
        
        return pd.to_datetime(f'{year}-{month}-{day}').date()
    else:
        return None

# Example usage:
data['date'] = [extract_date_from_url(url) for url in data['url'].values]

In [ ]:
data.to_csv("article_links_updated.csv", index=False)

In [ ]:
def extract_text(html_content):
    # Parse the HTML content
    soup = BeautifulSoup(str(html_content), "html.parser")
    # Extract and return the textual part of the HTML
    return soup.get_text(separator="\n")

In [ ]:
data['date'] = data['url'].apply(extract_date_from_url)

In [ ]:
articles_data = []

for url in data['url']:
    print(f"Scraping: {url}")
    article = scrapeArticle(url)
    if article:
        articles_data.append(article)

In [ ]:
articles_df = pd.DataFrame(articles_data)

In [ ]:
articles_df = articles_df.merge(
    data[['url', 'date']],
    left_on='Link',
    right_on='url',
    how='left'
).drop(columns='url')

In [ ]:
articles_df.to_csv("aljazeera_articles_full_1-10-24.csv", index=False)